# 🏋️‍♂️ Human-Object Interaction (HOI) Multimodal BiLSTM for Sports Action Recognition
### Master's / Bachelor's Thesis Research Pipeline

This notebook trains and evaluates the **HOI-BiLSTM** model across **82 sports categories**.

**Architecture Streams**:
1. **Visual Stream ($C_t$)**: Scene & appearance features (ResNet-50)
2. **Human Pose Stream ($H_t$)**: 17 anatomical keypoints + velocities (Keypoint R-CNN)
3. **Sports Object Stream ($O_t$)**: Equipment centroid, scale & velocity (Faster R-CNN)
4. **Interaction Stream ($I_t$)**: Geometric joint-to-object distances and approach rates
5. **Temporal BiLSTM + Attention**: 2-layer Bidirectional LSTM with Temporal Attention Pooling

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi

## 2. Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!git clone https://github.com/angkon101/Sport-Action-Detection.git
%cd /content/Sport-Action-Detection

## 3. Install Required Dependencies

In [ ]:
!pip install -r requirements.txt

## 4. Unpack the Sports Dataset from Google Drive
*(Upload `ucf_action.zip` to the root of your Google Drive first)*

In [ ]:
!mkdir -p 'ucf action'
!unzip -q '/content/drive/MyDrive/ucf_action.zip' -d 'ucf action/'

# Verify classes found
import os
classes = [d for d in os.listdir('ucf action') if os.path.isdir(os.path.join('ucf action', d))]
print(f'Successfully loaded {len(classes)} sport categories in Colab!')

## 5. Extract Multimodal Features (Visual + Pose + Object + HOI)
This runs the pretrained ResNet-50 + Keypoint R-CNN + Faster R-CNN on the videos and caches the compact `.pt` feature files to `data/features`.

In [ ]:
!python src/extractors/batch_extract_features.py --data-dir 'ucf action' --output-dir 'data/features' --num-frames 16

# Optional: Back up the extracted features to your Google Drive so you never need to extract again!
!zip -rq '/content/drive/MyDrive/hoi_features_cached.zip' data/features

## 6. Train the Proposed HOI-BiLSTM Model (Ablation M7 - Full Proposed Model)
Trains the 2-layer BiLSTM with Temporal Attention on all 82 sport classes.

In [ ]:
!python train_hoi.py --features-dir 'data/features' --model-mode M7 --epochs 40 --batch-size 32 --lr 0.0005

## 7. Run Thesis Ablation Experiments (M1 through M6)
Generate the exact comparison table for your thesis chapter by running the baseline ablations:

In [ ]:
# M1: Visual CNN + BiLSTM (Standard Baseline)
!python train_hoi.py --features-dir 'data/features' --model-mode M1 --epochs 30

# M4: Visual + Human Pose + BiLSTM
!python train_hoi.py --features-dir 'data/features' --model-mode M4 --epochs 30

# M6: Visual + Pose + Object + Interaction (Without Attention)
!python train_hoi.py --features-dir 'data/features' --model-mode M6 --epochs 30

## 8. Save Trained Checkpoints to Google Drive

In [ ]:
!cp -r checkpoints '/content/drive/MyDrive/Thesis_Checkpoints_HOIBiLSTM'